# [16.5] VLM Modality and Region SHAP

## Core question

When a VLM score improves, did the useful evidence come from the image, the text, the target object, an OCR-like region, or background leakage?

## Learning objectives

By the end, you should be able to:

1. Compute exact Shapley values from complete finite coalition tables.
2. Use efficiency as a hard invariant before interpreting modality or region scores.
3. Detect image/text synergy in a two-player VLM-style game.
4. Treat object, background, and OCR as structured region players.
5. Interpret a pinned CLIP rendered-image report without overclaiming.

> Difficulty: 4/5  
> Importance: 4/5

<img src="../../instructions/assets/vlm_modality_region_validation_loop.svg" width="760">

The toy modality game should produce values `[2.0, 1.5]`. The toy region game should produce values `[2.25, 0.0, 1.0]`. The pinned CLIP report then checks the same control logic on real logits from deterministic rendered images.

<details><summary>Help - why start with toy games?</summary>

VLM attribution has many plausible failure modes. Toy games make the target exact before we inspect real logits: image/text synergy, object evidence, OCR shortcuts, and background controls all have known expected behavior.

</details>


## Setup

Run the setup cell once. The visible tests are deterministic and small; the committed report cell reads the pinned CLIP CUDA result.

<details><summary>Expected output</summary>

No printed output. Imports should succeed and the report dataclasses should be defined.

</details>


In [ ]:
from collections.abc import Callable, Mapping
from dataclasses import dataclass
import itertools
import json
import math
import sys
from pathlib import Path

import torch as t

chapter = "chapter16_shapley_attribution_baselines"
section = "part5_vlm_modality_region_shap"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part5_vlm_modality_region_shap.tests as tests

Coalition = frozenset[int]
MAIN = True


@dataclass(frozen=True)
class ShapleyEfficiencyReport:
    shapley_sum: float
    total_value_delta: float
    efficiency_error: float
    satisfies_efficiency: bool


@dataclass(frozen=True)
class VLMModalitySHAPReport:
    modality_values: t.Tensor
    baseline_score: float
    image_only_score: float
    text_only_score: float
    full_score: float
    synergy: float
    detects_synergy: bool
    satisfies_efficiency: bool


@dataclass(frozen=True)
class VLMRegionSHAPReport:
    region_values: t.Tensor
    region_names: tuple[str, ...]
    target_region: str
    target_value: float
    max_background_value: float
    localizes_target: bool
    satisfies_efficiency: bool


## Exercise 1 - exact Shapley and efficiency

Implement exact weighted marginal contributions and the efficiency report.

<details><summary>Help - what does efficiency catch?</summary>

Efficiency says the attribution sum must equal full coalition value minus empty coalition value. If it fails, either the coalition table is incomplete, the baseline is wrong, or the weighting loop is wrong.

</details>

<details><summary>Expected output</summary>

```text
All tests in `test_exact_shapley_values_splits_two_player_synergy` passed!
All tests in `test_shapley_efficiency_report_requires_complete_coalition_table` passed!
```

</details>

<details><summary>Common bugs</summary>

- Using unweighted leave-one-out deltas.
- Forgetting the empty coalition.
- Letting incomplete tables pass silently.

</details>

<details><summary>Solution</summary>

Normalize keys to `frozenset`, require a complete table, then sum weighted marginal effects using the Shapley factorial weight.

</details>


In [ ]:
def all_coalitions(num_players: int) -> tuple[Coalition, ...]:
    """Return every player subset for a finite cooperative game."""
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


def normalize_coalition_values(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
) -> dict[Coalition, float]:
    """Normalize coalition keys and require a complete value table."""
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


def exact_shapley_values(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
) -> t.Tensor:
    """Compute exact Shapley values by summing weighted marginal effects."""
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


def shapley_efficiency_report(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
    tolerance: float = 1e-9,
) -> ShapleyEfficiencyReport:
    """Check that Shapley values sum to full coalition minus empty coalition."""
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


if MAIN:
    tests.test_exact_shapley_values_splits_two_player_synergy(exact_shapley_values)
    tests.test_shapley_efficiency_report_requires_complete_coalition_table(
        shapley_efficiency_report
    )


## Exercise 2 - modality SHAP

Build a two-player image/text game and detect the image/text synergy.

<details><summary>Help - what should the exact values be?</summary>

Image gets its additive `1.0` plus half of the two-point synergy, for `2.0`. Text gets its additive `0.5` plus the other half, for `1.5`.

</details>

<details><summary>Expected output</summary>

```text
All tests in `test_vlm_modality_game_contains_expected_image_text_coalitions` passed!
All tests in `test_vlm_modality_shap_report_detects_synergy_and_efficiency` passed!
```

</details>

<details><summary>What you should see</summary>

```text
modality_values = [2.0, 1.5]
synergy = 2.0
satisfies_efficiency = True
```

</details>

<details><summary>Common bugs</summary>

- Assigning all synergy to image.
- Passing weak synergy as a positive result.
- Hiding raw coalition scores.

</details>

<details><summary>Solution</summary>

Evaluate every image/text coalition, compute exact Shapley values, measure synergy, and require both synergy and efficiency.

</details>


In [ ]:
def coalition_values_from_function(
    num_players: int,
    value_fn: Callable[[Coalition], float],
) -> dict[Coalition, float]:
    """Evaluate `value_fn` on the complete coalition table."""
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


def vlm_modality_game(
    *,
    image_weight: float = 1.0,
    text_weight: float = 0.5,
    synergy_weight: float = 2.0,
) -> dict[Coalition, float]:
    """Return a two-player image/text game with multimodal synergy."""
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


def vlm_modality_shap_report(
    *,
    image_weight: float = 1.0,
    text_weight: float = 0.5,
    synergy_weight: float = 2.0,
    min_synergy: float = 1.0,
    tolerance: float = 1e-9,
) -> VLMModalitySHAPReport:
    """Compute modality Shapley values and detect image/text synergy."""
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


if MAIN:
    tests.test_vlm_modality_game_contains_expected_image_text_coalitions(
        vlm_modality_game
    )
    tests.test_vlm_modality_shap_report_detects_synergy_and_efficiency(
        vlm_modality_shap_report
    )


## Exercise 3 - region SHAP

Treat object, background, and OCR as structured region players. The background player is a negative control.

<details><summary>Help - why include OCR?</summary>

OCR is a plausible shortcut. This toy game gives OCR some value, but the object region must still beat OCR and background by a margin.

</details>

<details><summary>Expected output</summary>

```text
All tests in `test_vlm_region_game_keeps_background_as_negative_control` passed!
All tests in `test_vlm_region_shap_report_localizes_object_region` passed!
```

</details>

<details><summary>What you should see</summary>

```text
object      2.25
background  0.0
ocr_text    1.0
```

</details>

<details><summary>Common bugs</summary>

- Letting background pick up object credit.
- Misaligning `region_names` with attribution order.
- Checking only positive object value instead of object-over-control margin.

</details>

<details><summary>Solution</summary>

Use player `0` for object, `1` for zero-valued background, and `2` for OCR. Then compare the target object value against the strongest non-target value.

</details>


In [ ]:
def vlm_region_game(
    *,
    object_weight: float = 2.0,
    ocr_weight: float = 0.75,
    object_ocr_interaction: float = 0.5,
) -> dict[Coalition, float]:
    """Return a three-player object/background/OCR region game."""
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


def vlm_region_shap_report(
    *,
    region_names: tuple[str, ...] = ("object", "background", "ocr_text"),
    target_region: str = "object",
    min_margin: float = 0.5,
    tolerance: float = 1e-9,
) -> VLMRegionSHAPReport:
    """Compute structured region Shapley values and check target localization."""
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


if MAIN:
    tests.test_vlm_region_game_keeps_background_as_negative_control(
        vlm_region_game
    )
    tests.test_vlm_region_shap_report_localizes_object_region(
        vlm_region_shap_report
    )


## Exercise 4 - notebook contract

Expose JSON-serializable modality and region smoke reports.

<details><summary>Help - why use JSON-like dictionaries?</summary>

The committed report is JSON. Plain lists and numbers make notebook outputs easy to compare to the report and prevent hidden tensor serialization issues.

</details>

<details><summary>Expected output</summary>

```text
All tests in `test_modality_shap_smoke_test` passed!
All tests in `test_region_shap_smoke_test` passed!
All tests in `test_notebook_contract` passed!
```

</details>

<details><summary>Solution</summary>

Return a dictionary with `modality` and `region` reports.

</details>


In [ ]:
def _tensor_report(report) -> dict:
    result = report.__dict__.copy()
    for key, value in list(result.items()):
        if hasattr(value, "tolist"):
            result[key] = value.tolist()
    return result


def modality_shap_smoke_test() -> dict:
    return _tensor_report(vlm_modality_shap_report())


def region_shap_smoke_test() -> dict:
    return _tensor_report(vlm_region_shap_report())


def run_smoke_test(cpu: bool = True) -> dict:
    """Return JSON-like toy modality and region reports."""
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


if MAIN:
    tests.test_modality_shap_smoke_test(modality_shap_smoke_test)
    tests.test_region_shap_smoke_test(region_shap_smoke_test)
    tests.test_notebook_contract(run_smoke_test)


## Committed pinned CLIP report

Read the committed CUDA report. This is not a substitute for the toy exercises; it checks that the same controls survive real CLIP logits on deterministic rendered images.

<details><summary>Expected output</summary>

```text
preflight_passed: true
model_id: openai/clip-vit-base-patch32
modality_synergy: >= 2.0
object_margin: >= 1.0
target_distractor_margin: >= 2.0
```

</details>

<details><summary>Help - what does this prove?</summary>

It supports the mechanics of modality and region SHAP on a pinned rendered-image CLIP setup. It does not prove broad VLM attribution, natural-image localization, or pixel-level saliency.

</details>

<details><summary>Solution</summary>

Load `verification_report.json`, assert the accepted CUDA metrics, and return `metrics.gpu_test`.

</details>


In [ ]:
def load_committed_gpu_report() -> dict:
    report = json.loads((section_dir / "verification_report.json").read_text())
    gpu = report["metrics"]["gpu_test"]
    assert report["accepted"] is True and report["tests_passed"] is True
    assert gpu["cuda_available"] is True and gpu["preflight_passed"] is True
    tests.test_committed_gpu_report_records_real_clip_controls()
    return report


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = load_committed_gpu_report()["metrics"]["gpu_test"]
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


if MAIN:
    gpu_report = run_gpu_test()
    {
        "model_id": gpu_report["model_id"],
        "modality_synergy": gpu_report["modality_synergy"],
        "region_values": gpu_report["region_values"],
        "object_margin": gpu_report["object_margin"],
        "target_distractor_margin": gpu_report["target_distractor_margin"],
        "peak_vram_gb": gpu_report["peak_vram_gb"],
    }


## Signature Result

<img src="../../instructions/assets/vlm_modality_region_signature_result.svg" width="760">

The toy modality result is `[2.0, 1.5]`: image/text synergy is split fairly.
The toy region result is `[2.25, 0.0, 1.0]`: object evidence wins, background stays zero, and OCR is a smaller shortcut.
The pinned CLIP report repeats the control pattern with real logits.

<details><summary>Interpreting the result</summary>

The positive result is meaningful because it has exact toy ground truth, efficiency checks, object/background/OCR controls,
target-vs-distractor margin, and a pinned CUDA report.
If background or OCR beat the object region, this would be a negative localization result.

</details>

## Limitations

This does not prove VLM region SHAP on natural images, pixel-level saliency, causal visual-token flow, or absence of OCR shortcuts in real datasets.

## Bonus / Exploring Anomalies

Add misleading OCR text, random background texture, swapped captions, or a SigLIP checkpoint. Treat any new positive result as exploratory until it survives the same controls.
